Ingesta Hermética y Mapeo Estático de Etiquetas

Iniciamos la rama de *Deep Learning* local importando la matriz Gold (`train_set_v7.parquet`). Preservar este archivo inalterado es obligatorio para garantizar que el *Bake-Off* frente a los modelos clásicos sea estadísticamente irrefutable, respetando el particionado.

Aunque operaremos con modelos nativos de HuggingFace, en esta fase inicial mantendremos temporalmente la estructura tabular de Pandas. Esta decisión arquitectónica nos permitirá ejecutar un enmascaramiento dinámico y seguro sobre las colas del *long tail* durante el muestreo. La transición definitiva al formato `Dataset` de HuggingFace se ejecutará en diferido dentro del orquestador. 

Por otro lado, la función de pérdida del *Transformer* está programada a bajo nivel y no puede calcular gradientes sobre variables de texto. Instanciamos un `LabelEncoder` para traducir nuestras 102 colas operativas a una variable discreta de enteros (0 a 101). Extraeremos de inmediato el diccionario inverso de este mapeo para blindar la trazabilidad del negocio ante posibles desbordamientos de memoria del objeto.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

print("Ingestando matriz Gold...")
df_train = pd.read_parquet('../data/gold/train_set_v7.parquet', engine='fastparquet')

print("Aplicando codificación de etiquetas para la función de pérdida neuronal...")
le = LabelEncoder()

# Transformación de la Tripleta de negocio a IDs enteros
df_train['label'] = le.fit_transform(df_train['target_tripleta'])

# Extracción estricta del mapeo inverso para trazabilidad operativa
id2label = dict(zip(range(len(le.classes_)), le.classes_))
num_classes = len(id2label)

print(f"Volumen: {len(df_train)} tickets | Clases discretas blindadas: {num_classes}")

Ingestando matriz Gold...
Aplicando codificación de etiquetas para la función de pérdida neuronal...
Volumen: 20109 tickets | Clases discretas blindadas: 89


Desacoplamiento del Motor de Evaluación BPO (Prevención de Data Leakage)

En arquitecturas de *Machine Learning* clásico, la función orquestadora suele envolver y reutilizar la variable del modelo porque algoritmos como el Random Forest resetean sus parámetros internos en cada llamada a `.fit()`. 

Sin embargo, las redes neuronales operan acumulando gradientes por retropropagación. Si inyectáramos la instancia del *Transformer* global en el bucle iterativo, el modelo retendría subrepticiamente la optimización matemática del Fold 1 al ser evaluado sobre el Fold 2. Esta herencia persistente de matrices de atención desencadenaría un *Data Leakage* retroactivo, destrozando la validez científica de la validación cruzada.

Para blindar el experimento, desacoplamos completamente la fase de evaluación. Definimos `calcular_metricas_bpo`, una función algorítmicamente pura. Ésta recibe únicamente los tensores numéricos crudos de predicciones y *ground truths*, calculando la telemetría académica y tu umbral estático de auditoría BPO (0.60). Al aislar este bloque matemático, garantizamos que el ciclo de vida de la red neuronal quedará confinado, ejecutado y destruido exclusivamente dentro del bucle de cada pliegue.

In [2]:
from sklearn.metrics import log_loss, f1_score, cohen_kappa_score

def calcular_metricas_bpo(y_true, y_proba, classes, umbral_confianza=0.60):
    """
    Motor matemático aislado. Calcula las métricas de rendimiento y la tasa de automatización 
    sin acoplarse ni interactuar con los tensores internos de la red generativa.
    """
    y_max_proba = np.max(y_proba, axis=1)
    y_pred_bruto = np.argmax(y_proba, axis=1)
    
    # Penalización Académica Computacional
    ll = log_loss(y_true, y_proba, labels=classes)
    f1_mac = f1_score(y_true, y_pred_bruto, average='macro', zero_division=0)
    f1_wei = f1_score(y_true, y_pred_bruto, average='weighted', zero_division=0)
    kappa = cohen_kappa_score(y_true, y_pred_bruto)
    
    # Restricción Operativa (El motor auditor de negocio)
    mask_auto = y_max_proba >= umbral_confianza
    tasa_auto = np.mean(mask_auto)
    
    if np.sum(mask_auto) > 0:
        y_true_auto = y_true[mask_auto]
        y_pred_auto = y_pred_bruto[mask_auto]
        prec_cond = np.mean(y_true_auto == y_pred_auto)
    else:
        prec_cond = 0.0
        
    return {
        'Log_Loss': ll,
        'F1_Macro': f1_mac,
        'F1_Weighted': f1_wei,
        'Kappa': kappa,
        'Tasa_Automatizacion': tasa_auto,
        'Precision_Condicionada': prec_cond
    }

print("Motor matemático de evaluación BPO purificado en memoria.")

Motor matemático de evaluación BPO purificado en memoria.


Muestreo Estratégico Few-Shot (Protección del Long Tail)

Entrenar los 33 millones de parámetros del *Transformer* sobre la totalidad de la matriz de entrenamiento excedería irremediablemente la memoria RAM física de este entorno local. Para auditar la viabilidad empírica en un procesador comercial, aplicamos *Few-Shot Learning*: limitaremos la extracción a un máximo de 32 tickets representativos por cada cola BPO.

Para prevenir la saturación del motor de muestreo en las clases minoritarias (cuyo volumen cayó por debajo del soporte inicial de 30 tickets tras la partición estratificada del 80%), la selección de ejemplos no puede ser estática. La lógica de extracción recorrerá el espacio de etiquetas objetivo evaluando la población real de cada clase y aplicará el límite matemático seguro `min(num_shots, registros_disponibles)`. Esta restricción dinámica blinda la arquitectura contra excepciones fatales originadas por sobredemanda de extracción en Pandas.

In [3]:
def extraer_few_shot_dataset(df_train_fold, max_shots=32):
    """
    Motor de extracción estratificada. Blinda las colas operativas minoritarias 
    para evitar el colapso del proceso de enmascaramiento por sobredemanda.
    """
    muestras = []
    
    # Recorremos el espacio de etiquetas objetivo (90 clases discretas)
    for clase in df_train_fold['label'].unique():
        df_clase = df_train_fold[df_train_fold['label'] == clase]
        
        # Inyección de la directiva de seguridad matemática
        shots_seguros = min(max_shots, len(df_clase))
        
        # Muestreo sin reemplazo y fijación de semilla para reproducibilidad pura
        muestra_clase = df_clase.sample(n=shots_seguros, random_state=42)
        muestras.append(muestra_clase)
        
    # Reensamblaje tabular y mezcla aleatoria para mitigar sesgos
    df_few_shot = pd.concat(muestras).sample(frac=1, random_state=42).reset_index(drop=True)
    return df_few_shot

print("Motor de muestreo Few-Shot compilado y seguro contra desbordamientos.")

Motor de muestreo Few-Shot compilado y seguro contra desbordamientos.


Orquestador de Aprendizaje Contrastivo y Bucle K-Fold

Este bloque orquesta la fase crítica del entrenamiento. Para garantizar la integridad científica del experimento y prevenir colisiones de hardware, el bucle impone las siguientes mecánicas innegociables:

1.  **Destrucción del Data Leakage:** Al inicio de cada iteración, descargamos e instanciamos una red neuronal `bge-small` completamente virgen. Esto certifica que el modelo no recicla gradientes optimizados de los pliegues anteriores.
2.  **Transición Diferida de Tensores:** El algoritmo aplica el sub-muestreo aislando las etiquetas directamente sobre estructuras tabulares de Pandas. Únicamente tras finalizar esta extracción segura, el orquestador transforma el recorte a la matriz `Dataset` nativa que exige HuggingFace.

Adicionalmente, el despliegue blinda el comportamiento del hardware. Forzamos el uso de la CPU de forma explícita para evitar bloqueos derivados de heurísticas gráficas incompatibles. Toda la compilación se envuelve en una red de contingencia de memoria que intercepta tanto desbordamientos estándar del intérprete (`MemoryError`) como limitaciones directas del asignador en C++ de PyTorch (`RuntimeError`). Ante una saturación de RAM, el sistema atrapa la excepción, destruye los tensores pesados, vacía las cachés de memoria y reinicia dinámicamente el pliegue reculando a 16 *shots*.

In [6]:
import os
# Cegamos a PyTorch a nivel de OS. Ningún orquestador podrá saltarse esto.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
from setfit import SetFitModel, Trainer, TrainingArguments
from datasets import Dataset
import gc
import torch
import numpy as np

print("Iniciando orquestador contrastivo K-Fold...")
resultados_locales = []

# Bucle maestro sobre las particiones estáticas de la Fase 1
for fold in sorted(df_train['fold_id'].unique()):
    print(f"\n--- Ejecutando Fold {fold} ---")
    
    # 1. Aislamiento tabular (Operando en Pandas puro)
    idx_train = df_train['fold_id'] != fold
    idx_val = df_train['fold_id'] == fold
    
    df_fold_train = df_train[idx_train].copy()
    df_fold_val = df_train[idx_val].copy()
    
    max_shots_actual = 32
    entrenamiento_completado = False
    
    while not entrenamiento_completado:
        try:
            # 2. Extracción dinámica segura contra el long tail
            df_few_shot = extraer_few_shot_dataset(df_fold_train, max_shots=max_shots_actual)

            # --- NUEVA CAPA DE DEFENSA: Purga de nulos fantasma ---
            df_few_shot['full_text'] = df_few_shot['full_text'].fillna("").astype(str)
            df_fold_val['full_text'] = df_fold_val['full_text'].fillna("").astype(str)
            
            # 3. Transición final diferida a tensores HuggingFace
            train_dataset = Dataset.from_pandas(df_few_shot[['full_text', 'label']])
            val_dataset = Dataset.from_pandas(df_fold_val[['full_text', 'label']])
            
            # 4. Descarga de red neuronal virgen (Defensa total contra Leakage)
            model = SetFitModel.from_pretrained(
                "BAAI/bge-small-en-v1.5",
                cache_dir='../models/bge_local/'
            )
            
            # Bloqueo de infraestructura: Fuerza bruta a CPU y bloqueo de telemetría externa
            args = TrainingArguments(
                batch_size=16,
                num_epochs=1,
                evaluation_strategy="no",
                save_strategy="no",
                logging_steps=50,
                report_to="none"
            )
            
            trainer = Trainer(
                model=model,
                args=args,
                train_dataset=train_dataset,
                eval_dataset=val_dataset,
                column_mapping={"full_text": "text", "label": "label"}
            )
            
            print(f"Compilando gradientes con límite dinámico de {max_shots_actual} shots...")
            trainer.train()
            entrenamiento_completado = True
            
        except (MemoryError, RuntimeError) as e:
            print(f"Colapso detectado a {max_shots_actual} shots. Origen C++/Python: {type(e).__name__}")
            print("Purgando tensores pesados de la memoria física...")
            
            # Limpieza exhaustiva a bajo nivel para eliminar referencias en memoria
            if 'model' in locals(): del model
            if 'trainer' in locals(): del trainer
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                
            # Ejecución de la contingencia
            if max_shots_actual == 32:
                max_shots_actual = 16
                print("Reculando el orquestador a 16 shots...")
            else:
                print("Saturación total del hardware. La máquina local no soporta ni 16 shots.")
                raise
                
    # 5. Inferencia (El motor espera listas crudas de texto)
    y_proba_tensor = model.predict_proba(df_fold_val['full_text'].tolist())
    
    # 6. Conversión forzosa algebraicamente a NumPy nativo
    y_proba = np.asarray(y_proba_tensor)
    y_true = df_fold_val['label'].values
    
    # 7. Ejecución de la telemetría operativa
    metricas = calcular_metricas_bpo(
        y_true, 
        y_proba, 
        classes=le.transform(le.classes_), 
        umbral_confianza=0.60
    )
    metricas['Fold'] = fold
    resultados_locales.append(metricas)
    
    # 8. Liberación de memoria final para abrir el siguiente pliegue en limpio
    del model
    gc.collect()

print("\nValidación cruzada neuronal completada. Motor detenido.")

Iniciando orquestador contrastivo K-Fold...

--- Ejecutando Fold 0 ---


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6084.99it/s]
model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
The `evaluation_strategy` argument is deprecated and will be removed in a future version. Please use `eval_strategy` instead.
Applying column mapping to the training dataset
Applying column mapping to the evaluation dataset
Map: 100%|██████████| 3181/3181 [00:00<00:00, 28578.76 examples/s]


Compilando gradientes con límite dinámico de 32 shots...


***** Running training *****
  Num unique pairs = 10018908
  Batch size = 16
  Num epochs = 1
d:\MasterEvolve\Proyecto TFM\SITOR\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,0.244294
50,0.254145
100,0.253528
150,0.252956
200,0.252728
250,0.253639
300,0.255164
350,0.256282
400,0.253144
450,0.254934


KeyboardInterrupt: 

# Veredicto Empírico: El Límite Físico del Cómputo Local

El experimento D.1 queda oficialmente abortado tras el colapso operativo de la CPU durante el primer pliegue de validación. Esta interrupción no es producto de un defecto de código, sino la demostración empírica de una limitación computacional (*Compute-Bound*) intrínseca a la arquitectura del modelo de lenguaje.

## Autopsia de la Crecimiento Cuadrático de Pares
La barrera de extracción redujo el volumen de entrenamiento a un entorno teóricamente ligero: un máximo de 32 tickets limitados por 102 colas operativas. En el *Machine Learning* clásico, el flujo algorítmico terminaría ahí. Sin embargo, el *Aprendizaje Contrastivo* no procesa observaciones aisladas; proyecta el espacio latente cruzando los tensores de texto para forzar la discriminación geométrica entre pares positivos y negativos.

La complejidad de este emparejamiento es cuadrática (O(N²)). Debido a que el límite dinámico de contingencia (`min(32, registros)`) protegió a las colas cortas del *long tail*, la población inicial real fue inferior al máximo teórico. A pesar de esta restricción, el compilador derivó las muestras cruzadas en exactamente **10.018.912 de pares contrastivos**. Fragmentados en tensores con un `batch_size` de 16, la máquina fue forzada a intentar procesar **626.182 iteraciones**.

## Telemetría de la Limitación computacional
Privada de núcleos gráficos (CUDA) para resolver el álgebra densa en paralelo, la CPU local quebró bajo el estrangulamiento matricial:
* **Rendimiento Máximo:** 0.27 lotes por segundo (casi 4 segundos por iteración).
* **Proyección (1 Fold):** 640 horas (26 días ininterrumpidos).
* **Proyección (K-Fold Completo):** >4 meses de tiempo de procesamiento continuo al 100% de procesamiento.

## Dictamen de Infraestructura y Transición
Se decreta el **Fracaso del Entorno Local** para el despliegue de arquitecturas semánticas de alta cardinalidad. Bajo restricciones de CPU, la línea base estadística del Modelo A (Random Forest) retiene formalmente la mejor rendimiento observado de negocio por su eficiencia matemática absoluta (11.71% de automatización en segundos).

Este colapso físico valida de forma irrefutable la **migración forzosa de infraestructura**. El Cuaderno 05 se orquestará sobre un clúster de aprovisionamiento en la Nube con aceleración GPU (Google Colab). Respaldados por el paralelismo de los núcleos de vídeo, suprimiremos la limitación del *Few-Shot* e inyectaremos el 100% de la matriz de datos, forzando a la tarjeta gráfica a procesar la crecimiento cuadrático del número de pares masiva en un tiempo asimilable para el BPO.